# **Setup and Drive Mount**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# **Project Paths**

In [3]:
PROJECT_DIR = Path("/content/drive/MyDrive/مشروع تصميم نظم")

DATA_DIR = PROJECT_DIR
TRAIN_DIR = DATA_DIR / "Train"
VAL_DIR   = DATA_DIR / "Val"
TEST_DIR  = DATA_DIR / "Test"

for folder in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    print(folder, "exists:", folder.exists())

/content/drive/MyDrive/مشروع تصميم نظم/Train exists: True
/content/drive/MyDrive/مشروع تصميم نظم/Val exists: True
/content/drive/MyDrive/مشروع تصميم نظم/Test exists: True


# **Dataset Split Verification**

In [4]:
def count_files(split_dir):
    counts = {}
    for class_folder in sorted(split_dir.iterdir()):
        if class_folder.is_dir():
            files = list(class_folder.glob("*"))

            label = class_folder.name.strip()

            counts[label] = len(files)

    return counts

train_counts = count_files(TRAIN_DIR)
val_counts = count_files(VAL_DIR)
test_counts = count_files(TEST_DIR)

print("Train:", train_counts)
print("Val:", val_counts)
print("Test:", test_counts)

Train: {'bacteria': 20, 'cancer': 20, 'emergency': 20, 'help': 25, 'hospital': 25, 'need': 20, 'pregnancy': 20, 'swelling': 20, 'virus': 20, 'wound': 20}
Val: {'bacteria': 2, 'cancer': 2, 'emergency': 2, 'help': 2, 'hospital': 2, 'need': 3, 'pregnancy': 3, 'swelling': 3, 'virus': 3, 'wound': 3}
Test: {'bacteria': 3, 'cancer': 3, 'emergency': 3, 'help': 3, 'hospital': 3, 'need': 2, 'pregnancy': 2, 'swelling': 2, 'virus': 2, 'wound': 2}


# **Class Balance Summary**

In [5]:
classes = sorted(set(train_counts) | set(val_counts) | set(test_counts))

summary = pd.DataFrame({
    "Class": classes,
    "Train": [train_counts.get(c, 0) for c in classes],
    "Validation": [val_counts.get(c, 0) for c in classes],
    "Test": [test_counts.get(c, 0) for c in classes],
})

summary["Total"] = (
    summary["Train"] +
    summary["Validation"] +
    summary["Test"]
)

summary

,Class,Train,Validation,Test,Total
0,bacteria,20,2,3,25
1,cancer,20,2,3,25
2,emergency,20,2,3,25
3,help,25,2,3,30
4,hospital,25,2,3,30
5,need,20,3,2,25
6,pregnancy,20,3,2,25
7,swelling,20,3,2,25
8,virus,20,3,2,25
9,wound,20,3,2,25


# **Dataset Metadata Creation**

In [6]:
all_data = []

for split_name, split_dir in {
    "train": TRAIN_DIR,
    "validation": VAL_DIR,
    "test": TEST_DIR
}.items():

    for class_folder in split_dir.iterdir():

        if class_folder.is_dir():

            label = class_folder.name.strip()

            for file in class_folder.glob("*"):

                all_data.append({
                    "split": split_name,
                    "label": label,
                    "file_name": file.name,
                    "file_path": str(file)
                })

dataset_df = pd.DataFrame(all_data)

dataset_df.head()

,split,label,file_name,file_path
0,train,hospital,Hospital_heba_1.mov,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...
1,train,hospital,Hospital_heba_2.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...
2,train,hospital,Hospital_heba_3.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...
3,train,hospital,Hospital_heba_4.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...
4,train,hospital,Hospital_heba_5.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...


# **Dataset Statistics**

In [7]:
print("Total samples:", len(dataset_df))
print("\nClasses:")
print(dataset_df["label"].value_counts())

print("\nSplits:")
print(dataset_df["split"].value_counts())

Total samples: 260

Classes:
label
hospital     30
help         30
need         25
bacteria     25
virus        25
swelling     25
pregnancy    25
cancer       25
wound        25
emergency    25
Name: count, dtype: int64

Splits:
split
train         210
validation     25
test           25
Name: count, dtype: int64


# **Data Augmentation Metadata**

In [8]:
def augment_label(label):
    augmentations = [
        label + "_flip",
        label + "_noise",
        label + "_scaled"
    ]

    return random.choice(augmentations)

dataset_df["augmented_label"] = dataset_df.apply(
    lambda row:
    augment_label(row["label"])
    if row["split"] == "train"
    else row["label"],
    axis=1
)

dataset_df.head()

,split,label,file_name,file_path,augmented_label
0,train,hospital,Hospital_heba_1.mov,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...,hospital_flip
1,train,hospital,Hospital_heba_2.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...,hospital_noise
2,train,hospital,Hospital_heba_3.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...,hospital_flip
3,train,hospital,Hospital_heba_4.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...,hospital_scaled
4,train,hospital,Hospital_heba_5.MOV,/content/drive/MyDrive/مشروع تصميم نظم/Train/h...,hospital_noise


In [9]:
csv_output_path = "/content/drive/MyDrive/مشروع تصميم نظم/final_dataset_metadata.csv"

dataset_df.to_csv(csv_output_path, index=False)

print("Dataset metadata saved successfully!")
print("Saved to:", csv_output_path)

Dataset metadata saved successfully!
Saved to: /content/drive/MyDrive/مشروع تصميم نظم/final_dataset_metadata.csv


In [10]:
path = "/content/drive/MyDrive/مشروع تصميم نظم/features_csv"

print(os.listdir(path))

['hand_features.csv', 'hand_features.gsheet']


In [11]:
features_path = "/content/drive/MyDrive/مشروع تصميم نظم/features_csv/hand_features.csv"

df = pd.read_csv(features_path)

print(df.head())
print(df.shape)
print(df.columns)

          0         1             2         3         4         5         6  \
0  0.649412  0.635976  1.740370e-07  0.636078  0.651262 -0.014083  0.630959   
1  0.650438  0.635511  1.824359e-07  0.635785  0.651905 -0.013401  0.629771   
2  0.649122  0.627259  1.623336e-07  0.633823  0.645992 -0.011899  0.628597   
3  0.648650  0.628541  2.008529e-07  0.633265  0.647774 -0.014478  0.627721   
4  0.647570  0.629037  1.902581e-07  0.631928  0.648209 -0.013643  0.626431   

          7         8         9  ...        56        57        58        59  \
0  0.682568 -0.016301  0.627840  ...  0.013009  0.636762  0.746947  0.011925   
1  0.683940 -0.015574  0.626249  ...  0.014239  0.637468  0.747841  0.013651   
2  0.679443 -0.014846  0.624613  ...  0.003329  0.638841  0.741419  0.002700   
3  0.682131 -0.017307  0.623939  ...  0.011911  0.637299  0.744820  0.010889   
4  0.682660 -0.016413  0.622291  ...  0.010114  0.636486  0.744725  0.009100   

         60        61        62        63   

In [12]:
df = pd.read_csv('/content/drive/MyDrive/مشروع تصميم نظم/features_csv/hand_features.csv')


X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


save_dir = "/content/drive/MyDrive/مشروع تصميم نظم/model_ready_data"
os.makedirs(save_dir, exist_ok=True)


np.save(f"{save_dir}/X_train.npy", X_train)
np.save(f"{save_dir}/y_train.npy", y_train)

np.save(f"{save_dir}/X_val.npy", X_val)
np.save(f"{save_dir}/y_val.npy", y_val)

np.save(f"{save_dir}/X_test.npy", X_test)
np.save(f"{save_dir}/y_test.npy", y_test)

print("All datasets saved successfully!")

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

All datasets saved successfully!
X_train shape: (18452, 65)
X_val shape: (2306, 65)
X_test shape: (2307, 65)


# **Data Augmentation on Training Features**

In [13]:
def add_noise(X, noise_level=0.01):
    noise = np.random.normal(0, noise_level, X.shape)
    return X + noise

def scale_features(X, scale_range=(0.95, 1.05)):
    scale = np.random.uniform(scale_range[0], scale_range[1], size=(X.shape[0], 1))
    return X * scale

# Original training data
X_train_original = X_train.copy()
y_train_original = y_train.copy()

# Augmented versions
X_train_noise = add_noise(X_train_original)
X_train_scaled = scale_features(X_train_original)

# Combine original + augmented data
X_train_augmented = np.concatenate([
    X_train_original,
    X_train_noise,
    X_train_scaled
], axis=0)

y_train_augmented = np.concatenate([
    y_train_original,
    y_train_original,
    y_train_original
], axis=0)

print("Original X_train shape:", X_train_original.shape)
print("Augmented X_train shape:", X_train_augmented.shape)
print("Original y_train shape:", y_train_original.shape)
print("Augmented y_train shape:", y_train_augmented.shape)

Original X_train shape: (18452, 65)
Augmented X_train shape: (55356, 65)
Original y_train shape: (18452,)
Augmented y_train shape: (55356,)


In [14]:
save_dir = "/content/drive/MyDrive/مشروع تصميم نظم/model_ready_data"
os.makedirs(save_dir, exist_ok=True)

np.save(f"{save_dir}/X_train.npy", X_train_augmented)
np.save(f"{save_dir}/y_train.npy", y_train_augmented)

np.save(f"{save_dir}/X_val.npy", X_val)
np.save(f"{save_dir}/y_val.npy", y_val)

np.save(f"{save_dir}/X_test.npy", X_test)
np.save(f"{save_dir}/y_test.npy", y_test)

print("Augmented model-ready datasets saved successfully!")
print("X_train shape:", X_train_augmented.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

Augmented model-ready datasets saved successfully!
X_train shape: (55356, 65)
X_val shape: (2306, 65)
X_test shape: (2307, 65)
